# FloodScout — train the flood model

Trains the model FloodScout scores regions with, on Google Colab.

**What it learns from.** One row per Kenyan county per month, 1996–2021: rainfall for the 30 days ending that month (ERA5 reanalysis, at one point or averaged over the county), the El Niño and Indian Ocean Dipole indices (NOAA ONI and DMI), elevation, slope and county area, and whether the Dartmouth Flood Observatory recorded a large flood there in that window. Years before 1996 are left out because the archive barely recorded Kenyan floods then.

**What it outputs.** The probability that a large flood is recorded in a region in a 30-day window.

The training code is not in this notebook. It is `riskengine/ml/train.py` from the repository, uploaded in the bundle below, so Colab and the app always run the same code.

**Before you start**, build the bundle on your machine from the repository root:

```
docker compose run --rm web python manage.py build_training_set
tar -czf data/processed/colab_bundle.tar.gz data/processed/training_set.csv $(ls data/processed/training_set.meta.json 2>/dev/null) riskengine/__init__.py riskengine/ml/__init__.py riskengine/ml/train.py riskengine/ml/features.py riskengine/ml/calibration.py geodata/__init__.py geodata/services/__init__.py geodata/services/climate_indices.py
```

## 1. Match the app's scikit-learn

The saved model is a pickle. It only loads reliably under the scikit-learn version that wrote it, so this must match `requirements.txt`. If Colab asks to restart the session after this cell, restart and carry on from step 2.

In [ ]:
%pip install -q "scikit-learn==1.9.1"
import sklearn
print(sklearn.__version__)

## 2. Upload `colab_bundle.tar.gz`

It is in `data/processed/` in the repository.

In [ ]:
from google.colab import files

uploaded = files.upload()
bundle = next(iter(uploaded))
!tar -xzf "$bundle"
!ls data/processed riskengine/ml geodata/services

## 3. Train

Compares Logistic Regression, Random Forest and Gradient Boosting, holding out whole years at a time, against two baselines: always guessing the base rate, and a model that knows only the month. Then refits the best one on all the data and calibrates its probabilities if that helps on held-out years. Takes a few minutes.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, ".")
from geodata.services.climate_indices import fetch_dmi, fetch_oni
from riskengine.ml import train

artifact, metrics = train.train(Path("data/processed/training_set.csv"), fetch_oni(), fetch_dmi())
train.save(artifact, metrics)
print("Trained", artifact["version"])

## 4. How good is it?

All figures are on years the model did not see. **Average precision** is the one the winner is picked on. Compare it with both baselines: beating the base rate shows the model knows *something*; beating the season-only model shows rainfall, the ocean indices and terrain add something beyond "it is the rainy season". **Brier** is lower-is-better.

In [ ]:
import pandas as pd

print(f"{metrics['rows']} county-months from {metrics['period'][0][:4]}, {metrics['positives']} with a recorded flood "
      f"(base rate {metrics['base_rate']:.1%}); rainfall: {metrics['rain_source']}")
print(f"Selected {metrics['selected']}: average precision {metrics['lift_over_base_rate']}x the base rate, "
      f"{metrics['lift_over_season_only']}x the season-only model. Calibration kept: {metrics['calibration']}")
comparison = pd.DataFrame(metrics["comparison"]).T
for name, scores in metrics["baselines"].items():
    comparison.loc[f"({name})"] = pd.Series(scores)
comparison

How often a flood was actually recorded in each risk band, on held-out years, after calibration. The bands are only useful if this rises from Low to Very High, and `mean_predicted` should sit close to `observed_flood_rate`.

In [ ]:
pd.DataFrame(metrics["band_calibration"]).set_index("band")

Calibration on held-out years. ROC AUC barely moves (calibration does not change the ranking); the one with the lowest Brier is kept.

In [ ]:
pd.DataFrame(metrics["calibration_comparison"]).T

Average precision in each fold, with the years it held out. The pooled figures above hide how much the result depends on which years are held out; the spread here is the honest error bar.

In [ ]:
folds = pd.DataFrame(
    {name: scores["average_precision"] for name, scores in fold["scores"].items()}
    | {"years": ", ".join(map(str, fold["years"])), "floods": fold["positives"]}
    for fold in metrics["folds"]
)
display(folds)
pd.DataFrame(metrics["fold_summary"]).T

What happens if a probability above each band edge is treated as a warning. A false negative is a recorded flood with no warning.

In [ ]:
pd.DataFrame(metrics["threshold_errors"]).set_index("threshold")

The final test. The years from 2017 on were set aside, the model and calibration were chosen again on 1996 to 2016 alone, and 2017 to 2021 were scored once. This is the figure to trust most, and it is the least flattering. The first row is the model that choice picked; the others are shown for the record.

In [ ]:
test = metrics["final_test"]
print(f"{test['rows']} county-months, {test['positives']} floods, {test['test_years']}")
display(pd.DataFrame(
    {f"{test['selected']} (chosen)": test["scores"]} | test["other_candidates"] | test["baselines"]
).T)
pd.DataFrame(test["band_calibration"]).set_index("band")

The commit, data and seed this result came from. `git_commit` is empty on Colab, because the bundle is not a git checkout: note the commit the bundle was built from.

In [ ]:
metrics["provenance"]

What pushes the probability up or down (logistic regression only). Coefficients are per standard deviation of each input, so they can be compared with each other. `log_area` is a control for how the labels are made, not a cause of flooding; see `riskengine/ml/README.md`.

In [ ]:
pd.Series(metrics.get("standardised_coefficients", {}), name="coefficient").sort_values().to_frame()

## 5. Download the model

Put both files in `riskengine/ml/artifacts/` in the repository, replacing what is there. Then:

```
docker compose restart web
docker compose run --rm web python manage.py seed_rainfall
docker compose run --rm web python manage.py score_regions
docker compose run --rm web python manage.py flood_outlook
```

In [ ]:
files.download(str(train.ARTIFACT_PATH))
files.download(str(train.METRICS_PATH))